In [21]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
import re

REAL_CSV = "preds_of_64Neurons_denseLayer_test.csv"
FAKE_CSV = "[img+obj_labels_to_fakes]fake_activations(test).csv"
CONFIRMED_CSV = "verification_summary_1 (test).csv" 


ID_COL = "filenames"


FILTER_MODE = "none"

FIRE_FRAC = 0.80
OUT_CSV = "per_neuron_suppression_(img+obj).csv"

In [22]:
real_df = pd.read_csv(REAL_CSV)
fake_df = pd.read_csv(FAKE_CSV)
fake_df["filenames"] = fake_df["filenames"].str.replace("fake_", "", regex=False)

if ID_COL is not None and ID_COL in real_df.columns and ID_COL in fake_df.columns:
    real_df = real_df.sort_values(ID_COL).reset_index(drop=True)
    fake_df = fake_df.sort_values(ID_COL).reset_index(drop=True)

    if not real_df[ID_COL].equals(fake_df[ID_COL]):
        raise ValueError("Real/Fake rows do not align by filename. Fix pairing first.")
else:
    if len(real_df) != len(fake_df):
        raise ValueError("Real/Fake have different row counts; cannot assume pairing.")


In [23]:
exclude = {ID_COL} if ID_COL is not None else set()

common_cols = [c for c in real_df.columns if c in fake_df.columns and c not in exclude]
neuron_cols = [c for c in common_cols if pd.api.types.is_numeric_dtype(real_df[c])]

if len(neuron_cols) == 0:
    raise ValueError("No numeric neuron columns found in both CSVs.")

print("Neuron columns found:", len(neuron_cols))


Neuron columns found: 64


In [24]:
conf_df = pd.read_csv(CONFIRMED_CSV)

def extract_int(x):
    m = re.search(r"\d+", str(x))
    return int(m.group()) if m else None

confirmed_cols = []

# If confirmed CSV contains actual column names:
for cand in ["colname", "column", "neuron_col", "neuron_name", "neuron"]:
    if cand in conf_df.columns:
        vals = conf_df[cand].dropna().astype(str).tolist()
        confirmed_cols = [v for v in vals if v in neuron_cols]
        if confirmed_cols:
            break

# Else map integer neuron ids to columns:
if not confirmed_cols:
    for cand in ["neuron_id", "id", "neuron"]:
        if cand in conf_df.columns:
            ids = [extract_int(v) for v in conf_df[cand].dropna().tolist()]
            ids = [i for i in ids if i is not None]

            def col_id(col): 
                return extract_int(col)

            confirmed_cols = [c for c in neuron_cols if col_id(c) in set(ids)]
            if confirmed_cols:
                break

confirmed_set = set(confirmed_cols)
print("Confirmed neurons found:", len(confirmed_set))


Confirmed neurons found: 25


In [25]:
R = real_df[neuron_cols].to_numpy(dtype=float)   # shape: (N_images, N_neurons)
F = fake_df[neuron_cols].to_numpy(dtype=float)

max_real = R.max(axis=0)                          # per neuron max over real images
thr = FIRE_FRAC * max_real                        # per neuron threshold (real-derived)


In [26]:
rows = []
N_images, N_neurons = R.shape

for j, col in enumerate(neuron_cols):
    r = R[:, j]
    f = F[:, j]

    # Build mask based on filter mode
    if FILTER_MODE == "none":
        mask = np.ones_like(r, dtype=bool)

    elif FILTER_MODE == "nonzero":
        mask = (r > 0) & (f > 0)

    elif FILTER_MODE == "real_fire":
        # only keep cases where neuron strongly fires in real
        if max_real[j] <= 0:
            mask = np.zeros_like(r, dtype=bool)
        else:
            mask = (r >= thr[j])

    else:
        raise ValueError(f"Unknown FILTER_MODE: {FILTER_MODE}")

    rr = r[mask]
    ff = f[mask]

    n_used = rr.size
    if n_used == 0:
        rows.append({
            "neuron": col,
            "is_confirmed": col in confirmed_set,
            "N_used": 0,
            "mean_drop": np.nan,
            "median_drop": np.nan,
            "prop_R_gt_F": np.nan,
            "median_pct_drop": np.nan,
        })
        continue

    diff = rr - ff

    # prop_R_gt_F = fraction of strict wins among non-ties
    non_tie = diff != 0
    if non_tie.sum() == 0:
        prop = np.nan
    else:
        prop = (diff[non_tie] > 0).mean()

    # percent drop only meaningful where real>0
    pos_real = rr > 0
    if pos_real.sum() == 0:
        med_pct = np.nan
    else:
        pct_drop = (rr[pos_real] - ff[pos_real]) / rr[pos_real]
        med_pct = np.median(pct_drop)

    rows.append({
        "neuron": col,
        "is_confirmed": col in confirmed_set,
        "N_used": int(n_used),
        "mean_drop": float(np.mean(diff)),
        "median_drop": float(np.median(diff)),
        "prop_R_gt_F": float(prop) if prop == prop else np.nan,  # keep NaN if nan
        "median_pct_drop": float(med_pct) if med_pct == med_pct else np.nan,
    })

supp_df = pd.DataFrame(rows)


In [27]:
print("\n=== Per-neuron suppression magnitude ===")
print("Filter mode:", FILTER_MODE)

# Sort by strongest suppression (mean_drop descending)
supp_sorted = supp_df.sort_values(["mean_drop"], ascending=False)

print("\nTop 10 most suppressed neurons (by mean_drop = mean(real-fake)):")
print(supp_sorted.head(10)[["neuron","is_confirmed","N_used","mean_drop","median_drop","prop_R_gt_F","median_pct_drop"]])

conf_only = supp_df[supp_df["is_confirmed"] == True].sort_values("mean_drop", ascending=False)
print("\nTop confirmed neurons (by mean_drop):")
print(conf_only.head(10)[["neuron","N_used","mean_drop","median_drop","prop_R_gt_F","median_pct_drop"]])



=== Per-neuron suppression magnitude ===
Filter mode: none

Top 10 most suppressed neurons (by mean_drop = mean(real-fake)):
   neuron  is_confirmed  N_used  mean_drop  median_drop  prop_R_gt_F  \
6       6         False     793   0.595307          0.0     0.700565   
9       9          True     793   0.594476          0.0     0.779762   
28     28          True     793   0.499519          0.0     0.722772   
39     39         False     793   0.422711          0.0     0.728070   
17     17         False     793   0.421513          0.0     0.761538   
44     44         False     793   0.411656          0.0     0.685841   
14     14         False     793   0.368699          0.0     0.722826   
19     19          True     793   0.368139          0.0     0.695652   
29     29          True     793   0.329409          0.0     0.661638   
0       0          True     793   0.309362          0.0     0.634731   

    median_pct_drop  
6          0.238296  
9          0.605386  
28         0.37

In [28]:
supp_df.to_csv(OUT_CSV, index=False)
print("\nSaved:", OUT_CSV)



Saved: per_neuron_suppression_(img+obj).csv


In [29]:
# A neuron is suppressed if average(real - fake) > 0
suppressed = supp_df[supp_df["mean_drop"] > 0].copy()

print("=== Suppressed Neurons ===")
print("Total suppressed neurons:", len(suppressed), "/", len(supp_df))

# Sort by strongest suppression
suppressed = suppressed.sort_values("mean_drop", ascending=False)

print("\nSuppressed neurons and their magnitude:")
print(
    suppressed[[
        "neuron",
        "mean_drop"
    ]].reset_index(drop=True)
)


=== Suppressed Neurons ===
Total suppressed neurons: 53 / 64

Suppressed neurons and their magnitude:
   neuron  mean_drop
0       6   0.595307
1       9   0.594476
2      28   0.499519
3      39   0.422711
4      17   0.421513
5      44   0.411656
6      14   0.368699
7      19   0.368139
8      29   0.329409
9       0   0.309362
10     57   0.295876
11     18   0.294935
12     13   0.285758
13     52   0.283947
14      1   0.245806
15     22   0.240611
16     41   0.230780
17     24   0.223557
18     38   0.217006
19     46   0.213259
20     63   0.204827
21      7   0.197680
22     61   0.189757
23     47   0.187040
24     51   0.186787
25     36   0.183905
26     23   0.180875
27     31   0.174096
28     50   0.165152
29     15   0.164563
30     42   0.163779
31     34   0.155671
32     54   0.148218
33     10   0.140644
34     56   0.139395
35     37   0.138030
36     58   0.133008
37      2   0.108219
38     33   0.101949
39     48   0.095475
40     25   0.074716
41     40   0.06

In [30]:
conf_only = supp_df[supp_df["is_confirmed"] == True].sort_values("mean_drop", ascending=False)
print("\nTop confirmed neurons (by mean_drop):")
print(conf_only.head(10)[["neuron","N_used","mean_drop","median_drop","prop_R_gt_F","median_pct_drop"]])


Top confirmed neurons (by mean_drop):
   neuron  N_used  mean_drop  median_drop  prop_R_gt_F  median_pct_drop
9       9     793   0.594476          0.0     0.779762         0.605386
28     28     793   0.499519          0.0     0.722772         0.370117
19     19     793   0.368139          0.0     0.695652         0.433473
29     29     793   0.329409          0.0     0.661638         0.387435
0       0     793   0.309362          0.0     0.634731         0.517421
41     41     793   0.230780          0.0     0.585799         0.311316
46     46     793   0.213259          0.0     0.566148         0.258755
7       7     793   0.197680          0.0     0.569405         0.521848
61     61     793   0.189757          0.0     0.616967         0.373798
36     36     793   0.183905          0.0     0.551887         0.288650


In [31]:
# A neuron is suppressed if average(real - fake) > 0
suppressed = conf_only[conf_only["mean_drop"] > 0].copy()

print("=== Suppressed Neurons ===")
print("Total suppressed neurons:", len(suppressed), "/", len(conf_only))

# Sort by strongest suppression
suppressed = suppressed.sort_values("mean_drop", ascending=False)

print("\nSuppressed neurons and their magnitude:")
print(
    suppressed[[
        "neuron",
        "mean_drop"
    ]].reset_index(drop=True)
)


=== Suppressed Neurons ===
Total suppressed neurons: 21 / 25

Suppressed neurons and their magnitude:
   neuron  mean_drop
0       9   0.594476
1      28   0.499519
2      19   0.368139
3      29   0.329409
4       0   0.309362
5      41   0.230780
6      46   0.213259
7       7   0.197680
8      61   0.189757
9      36   0.183905
10     31   0.174096
11     42   0.163779
12     58   0.133008
13     48   0.095475
14     40   0.065504
15     62   0.057268
16     16   0.052203
17     43   0.040709
18     49   0.036457
19     35   0.034012
20     60   0.003343
